# 01 — Fetch Asset Prices
**GeoSentinel Terminal (VARTA) · Team 7 Lambda · SP2026**

Fetches daily Open-High-Low-Close-Volume (OHLCV) price data for all 12 locked assets via yfinance.
Window: `DATE_TRAIN_START` (2010-08-01) → `DATE_TRAIN_END` (2024-12-31).
All 12 tickers have clean data across this full window — no null values expected.

Outputs: `data/processed/prices.parquet`

In [1]:
# ── Path setup — run from any directory ──────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath(".."))

In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
import polars as pl
from config import TICKERS, DATE_TRAIN_START, DATE_TRAIN_END, DATA_PROC
from src.utils import log, save_parquet

log.info(f"Fetching {len(TICKERS)} tickers: {TICKERS}")
log.info(f"Window: {DATE_TRAIN_START} → {DATE_TRAIN_END}")

2026-04-15 15:45:01 [INFO] varta — Fetching 12 tickers: ['REMX', 'LIT', 'ALB', 'FCX', 'NVDA', 'TSM', 'AMD', 'BNO', 'XOM', 'CVX', 'GLD', 'SPY']


2026-04-15 15:45:01 [INFO] varta — Window: 2010-08-01 → 2024-12-31


In [3]:
# ── Fetch raw OHLCV from yfinance ─────────────────────────────────────────────
raw = yf.download(
    TICKERS,
    start=DATE_TRAIN_START,
    end=DATE_TRAIN_END,
    auto_adjust=True,
    progress=True,
)
print(f"Raw shape: {raw.shape}")
raw.tail(3)

[                       0%                       ]

[********              17%                       ]  2 of 12 completed

[********              17%                       ]  2 of 12 completed

[********************  42%                       ]  5 of 12 completed

[**********************58%***                    ]  7 of 12 completed

...

[**********************83%***************        ]  10 of 12 completed

[**********************92%*******************    ]  11 of 12 completed

[*********************100%***********************]  12 of 12 completed

Raw shape: (3628, 60)


Price           Close                                                \
Ticker            ALB         AMD        BNO         CVX        FCX   
Date                                                                  
2024-12-26  87.121941  125.059998  29.350000  136.268204  38.505489   
2024-12-27  86.281494  125.190002  29.629999  136.287140  38.181255   
2024-12-30  84.180405  122.440002  29.920000  135.406952  37.493481   

Price                                                                 ...  \
Ticker             GLD        LIT        NVDA       REMX         SPY  ...   
Date                                                                  ...   
2024-12-26  243.070007  41.955627  139.884171  39.577621  592.741577  ...   
2024-12-27  241.399994  41.579170  136.965118  39.224072  586.502014  ...   
2024-12-30  240.630005  41.281040  137.444946  38.654472  579.809204  ...   

Price       Volume                                                           \
Ticker         BNO      CVX       FCX      GLD     LIT       NVDA      REMX   
Date                                                                          
2024-12-26  155500  4492600   6127100  4645100  251700  116205600   61500.0   
2024-12-27  172800  5296500   7892500  4728100  250600  170582600   50600.0   
2024-12-30  230000  6194800  11047300  3522500  330100  167734700  138000.0   

Price                                     
Ticker           SPY       TSM       XOM  
Date                                      
2024-12-26  41219100   8044900   9652400  
2024-12-27  64969300  10664400  11943900  
2024-12-30  56578800  11235200  11080800  

[3 rows x 60 columns]

In [4]:
# ── Reshape MultiIndex → long format Polars DataFrame ────────────────────────
# yfinance returns MultiIndex columns: (field, ticker)
frames = []
for ticker in TICKERS:
    try:
        df_t = raw.xs(ticker, axis=1, level=1).reset_index()
        df_t.columns = [c.lower().replace(" ", "_") for c in df_t.columns]
        if "price" in df_t.columns:
            df_t = df_t.rename(columns={"price": "date"})
        if "date" not in df_t.columns:
            df_t = df_t.rename(columns={df_t.columns[0]: "date"})
        df_t["ticker"] = ticker
        df_t["return_1d"] = df_t["close"].pct_change()
        frames.append(df_t)
        log.info(f"  {ticker}: {len(df_t):,} rows")
    except Exception as e:
        log.warning(f"  {ticker} failed: {e}")

combined_pd = pd.concat(frames, ignore_index=True)
df = (
    pl.from_pandas(combined_pd)
    .select(["date", "ticker", "open", "high", "low", "close", "volume", "return_1d"])
    .filter(pl.col("close").is_not_null())   # drop pre-inception nulls (e.g. REMX Aug–Sep 2010)
)
print(f"Long format: {df.shape}")
df.head(5)

2026-04-15 15:45:03 [INFO] varta —   REMX: 3,628 rows


2026-04-15 15:45:03 [INFO] varta —   LIT: 3,628 rows


2026-04-15 15:45:03 [INFO] varta —   ALB: 3,628 rows


2026-04-15 15:45:03 [INFO] varta —   FCX: 3,628 rows


2026-04-15 15:45:03 [INFO] varta —   NVDA: 3,628 rows


...

2026-04-15 15:45:03 [INFO] varta —   CVX: 3,628 rows


2026-04-15 15:45:03 [INFO] varta —   GLD: 3,628 rows


2026-04-15 15:45:03 [INFO] varta —   SPY: 3,628 rows


Long format: (43474, 8)


date,ticker,open,high,low,close,volume,return_1d
datetime[ms],str,f64,f64,f64,f64,f64,f64
2010-10-28 00:00:00,"""REMX""",157.086754,157.086754,149.50061,149.50061,182742.0,null
2010-10-29 00:00:00,"""REMX""",154.634694,158.389445,153.332018,157.929688,140775.0,0.056382
2010-11-01 00:00:00,"""REMX""",161.914336,162.067595,155.554242,157.086792,78042.0,-0.005337
2010-11-02 00:00:00,"""REMX""",159.308971,159.308971,155.171079,156.703629,73917.0,-0.002439
2010-11-03 00:00:00,"""REMX""",157.240039,157.240039,153.485278,155.55423,45950.0,-0.007335


In [5]:
# ── Validate ──────────────────────────────────────────────────────────────────
passed = True

# 1. All 12 tickers present
present = set(df["ticker"].unique().to_list())
missing_tickers = set(TICKERS) - present
if missing_tickers:
    log.warning(f"Missing tickers: {missing_tickers}")
    passed = False
else:
    log.info(f"✓ All {len(TICKERS)} tickers present")

# 2. Date range
min_date = df["date"].min()
max_date = df["date"].max()
log.info(f"Date range: {min_date} → {max_date}")

# 3. Null check on close
null_close = df["close"].null_count()
if null_close > 0:
    log.warning(f"Null close prices: {null_close}")
    passed = False
else:
    log.info("✓ No null close prices")

# 4. Row count per ticker (expect ~3,600 trading days across 14 years)
counts = df.group_by("ticker").agg(pl.len().alias("n_rows")).sort("ticker")
print(counts)

print(f"\nValidation passed: {passed}")

2026-04-15 15:45:03 [INFO] varta — ✓ All 12 tickers present


2026-04-15 15:45:03 [INFO] varta — Date range: 2010-08-02 00:00:00 → 2024-12-30 00:00:00


2026-04-15 15:45:03 [INFO] varta — ✓ No null close prices


shape: (12, 2)
┌────────┬────────┐
│ ticker ┆ n_rows │
│ ---    ┆ ---    │
│ str    ┆ u32    │
╞════════╪════════╡
│ ALB    ┆ 3628   │
│ AMD    ┆ 3628   │
│ BNO    ┆ 3628   │
│ CVX    ┆ 3628   │
│ FCX    ┆ 3628   │
│ …      ┆ …      │
│ NVDA   ┆ 3628   │
│ REMX   ┆ 3566   │
│ SPY    ┆ 3628   │
│ TSM    ┆ 3628   │
│ XOM    ┆ 3628   │
└────────┴────────┘

Validation passed: True


In [6]:
# ── Save to Parquet ────────────────────────────────────────────────────────────
assert passed, "Validation failed — fix issues above before saving"
save_parquet(df, DATA_PROC / "prices.parquet", "asset prices")
print("Saved → data/processed/prices.parquet")

2026-04-15 15:45:03 [INFO] varta — Saved asset prices → /Users/taruntheegela/Desktop/VARTA/data/processed/prices.parquet (43,474 rows)


Saved → data/processed/prices.parquet


In [7]:
# ── Quick preview ─────────────────────────────────────────────────────────────
import plotly.express as px

df_plot = df.filter(pl.col("ticker").is_in(["SPY", "GLD", "NVDA", "BNO"]))
pd_plot = df_plot.to_pandas()

# Rebase to 100 at start
pd_plot["close_norm"] = pd_plot.groupby("ticker")["close"].transform(lambda x: x / x.iloc[0] * 100)

fig = px.line(pd_plot, x="date", y="close_norm", color="ticker",
              title="Selected Assets — Rebased to 100 (2010-08-01)",
              template="plotly_dark")
fig.show()